# NumPy 100 — Problems 31–40

Questions from [rougier/numpy-100](https://github.com/rougier/numpy-100) (MIT License). Solutions below are my own.

**Workflow:** try 10–15 min → docs → hints → official solution → rewrite from memory.

In [1]:
import numpy as np

---
### 31. How to ignore all numpy warnings (not recommended)? (★☆☆)

In [2]:
# Best practice: a context manager that ignores warnings only inside the block
with np.errstate(all="ignore"):
    Z = np.ones(1) / 0
print(Z)                                # [inf]

# Global switch (what the question asks) -- save the old settings so you can restore them
defaults = np.seterr(all="ignore")
Z = np.ones(1) / 0                      # no warning now
_ = np.seterr(**defaults)               # restore the defaults

# Only silence one kind of warning:
with np.errstate(divide="ignore"):
    Z = np.ones(1) / 0

[inf]


**Concept:** `np.seterr(all="ignore")` turns off NumPy's floating-point warnings globally and returns the old settings, so you can restore them with `np.seterr(**defaults)`. Prefer `with np.errstate(...)`, which limits the change to one block. Ignoring everything is not recommended because warnings often point to real bugs.

**Needed help?** hint  
**Revisit on:** Nov 3

---
### 32. Is the following expressions true? (★☆☆)
```python
np.sqrt(-1) == np.emath.sqrt(-1)
```

In [3]:
with np.errstate(all="ignore"):         # np.sqrt(-1) normally emits a RuntimeWarning
    a = np.sqrt(-1)
b = np.emath.sqrt(-1)

print(a)                                # nan
print(b)                                # 1j
print(a == b)                           # False

nan
1j
False


**Concept:** `np.sqrt(-1)` stays in the real numbers and returns `nan`; `np.emath.sqrt(-1)` switches to complex numbers and returns `1j`. `nan == 1j` is False (and `nan` is never equal to anything), so the expression is **False**.

**Needed help?** solution  
**Revisit on:** Nov 3

---
### 33. How to get the dates of yesterday, today and tomorrow? (★☆☆)

In [4]:
yesterday = np.datetime64("today") - np.timedelta64(1, "D")
today     = np.datetime64("today")
tomorrow  = np.datetime64("today") + np.timedelta64(1, "D")
print(yesterday, today, tomorrow)       # these dates depend on the day you run it

# with a fixed date, the result is always the same:
d = np.datetime64("2026-10-29")
print(d - np.timedelta64(1, "D"), d, d + np.timedelta64(1, "D"))

2026-11-03 2026-11-04 2026-11-05
2026-10-28 2026-10-29 2026-10-30


**Concept:** `np.datetime64("today")` gives today's date (day precision). Add or subtract a `np.timedelta64(n, "D")` to move by days. The first printed line changes every time you run it; the second line uses a fixed date.

**Needed help?** hint  
**Revisit on:** Nov 3

---
### 34. How to get all the dates corresponding to the month of July 2016? (★★☆)

In [5]:
Z = np.arange("2016-07", "2016-08", dtype="datetime64[D]")
print(Z)
print(len(Z))                           # 31 days
print(Z[0], Z[-1])                      # 2016-07-01 2016-07-31

['2016-07-01' '2016-07-02' '2016-07-03' '2016-07-04' '2016-07-05'
 '2016-07-06' '2016-07-07' '2016-07-08' '2016-07-09' '2016-07-10'
 '2016-07-11' '2016-07-12' '2016-07-13' '2016-07-14' '2016-07-15'
 '2016-07-16' '2016-07-17' '2016-07-18' '2016-07-19' '2016-07-20'
 '2016-07-21' '2016-07-22' '2016-07-23' '2016-07-24' '2016-07-25'
 '2016-07-26' '2016-07-27' '2016-07-28' '2016-07-29' '2016-07-30'
 '2016-07-31']
31
2016-07-01 2016-07-31


**Concept:** `np.arange` works with dates when you set `dtype="datetime64[D]"`. The start (`"2016-07"`) is inclusive and the stop (`"2016-08"`) is exclusive, so you get every day of July. The `[D]` unit means one step per day.

**Needed help?** solution  
**Revisit on:** Nov 3

---
### 35. How to compute ((A+B)*(-A/2)) in place (without copy)? (★★☆)

In [6]:
A = np.array([1.0, 2.0, 3.0])
B = np.array([4.0, 5.0, 6.0])

expected = (A + B) * (-A / 2)           # normal way (creates temporary arrays), for checking only
id_A = id(A)

np.add(A, B, out=B)                     # B = A + B
np.divide(A, 2, out=A)                  # A = A / 2
np.negative(A, out=A)                   # A = -A / 2
np.multiply(A, B, out=A)                # A = (-A / 2) * (A + B)

print(A)                                # [ -2.5  -7.  -13.5]
print(np.allclose(A, expected))         # True
print(id(A) == id_A)                    # True: same array object, no copy

[ -2.5  -7.  -13.5]
True
True


**Concept:** ufuncs accept an `out=` argument that writes the result into an existing array instead of allocating a new one. Chain `np.add`, `np.divide`, `np.negative`, `np.multiply` with `out=` to avoid temporary arrays. The arrays must be float so `/ 2` can be stored back. Note that `B` is overwritten with `A + B` along the way.

**Needed help?** solution  
**Revisit on:** Nov 3

---
### 36. Extract the integer part of a random array of positive numbers using 4 different methods (★★☆)

In [7]:
rng = np.random.default_rng(42)
Z = rng.uniform(0, 10, 10)
print(Z)

print(Z - Z % 1)                # 1: subtract the fractional part
print(np.floor(Z))              # 2: round down
print(Z.astype(int))            # 3: cast to int (drops the decimals)
print(np.trunc(Z))              # 4: cut off the decimals

# all four agree for positive numbers:
print(np.array_equal(Z - Z % 1, np.floor(Z)),
      np.array_equal(np.floor(Z), Z.astype(int)),
      np.array_equal(np.floor(Z), np.trunc(Z)))

[7.73956049 4.3887844  8.5859792  6.97368029 0.94177348 9.75622352
 7.61139702 7.86064305 1.28113633 4.50385938]
[7. 4. 8. 6. 0. 9. 7. 7. 1. 4.]
[7. 4. 8. 6. 0. 9. 7. 7. 1. 4.]
[7 4 8 6 0 9 7 7 1 4]
[7. 4. 8. 6. 0. 9. 7. 7. 1. 4.]
True True True


**Concept:** for positive numbers, `Z - Z % 1`, `np.floor(Z)`, `Z.astype(int)` and `np.trunc(Z)` all give the integer part. `astype(int)` and `trunc` cut toward zero, while `floor` rounds down; they only differ for negative numbers (`-2.5` gives `-3` with floor but `-2` with trunc). `np.ceil(Z) - 1` also looks similar but is wrong for values that are already whole numbers.

**Needed help?** hint  
**Revisit on:** Nov 8

---
### 37. Create a 5x5 matrix with row values ranging from 0 to 4 (★★☆)

In [8]:
Z = np.zeros((5, 5))
Z += np.arange(5)               # broadcasting: the length-5 row is added to every row
print(Z)

# same result with tile:
print(np.tile(np.arange(5), (5, 1)))

[[0. 1. 2. 3. 4.]
 [0. 1. 2. 3. 4.]
 [0. 1. 2. 3. 4.]
 [0. 1. 2. 3. 4.]
 [0. 1. 2. 3. 4.]]
[[0 1 2 3 4]
 [0 1 2 3 4]
 [0 1 2 3 4]
 [0 1 2 3 4]
 [0 1 2 3 4]]


**Concept:** broadcasting. Adding a shape `(5,)` array to a `(5, 5)` array repeats the row down every row automatically, with no loop. `np.tile(row, (5, 1))` builds the same matrix by repeating the row.

**Needed help?** no  
**Revisit on:** not needed

---
### 38. Consider a generator function that generates 10 integers and use it to build an array (★☆☆)

In [9]:
def generate():
    for x in range(10):
        yield x

Z = np.fromiter(generate(), dtype=int, count=10)
print(Z)
print(Z.dtype)

# a generator is used up after one pass, so np.array(generate()) would NOT work;
# use np.fromiter (or np.array(list(generate())))

[0 1 2 3 4 5 6 7 8 9]
int64


**Concept:** `np.fromiter(iterable, dtype, count)` builds an array directly from a generator without first creating a list. The `dtype` is required, and `count` is optional but lets NumPy pre-allocate memory. `np.array(generator)` does not work because it would create a 0-d object array.

**Needed help?** solution  
**Revisit on:** Nov 8

---
### 39. Create a vector of size 10 with values ranging from 0 to 1, both excluded (★★☆)

In [10]:
Z = np.linspace(0, 1, 11, endpoint=False)[1:]   # drop the leading 0, exclude the end
print(Z)
print(len(Z), Z.min() > 0, Z.max() < 1)          # 10 True True

# alternative: make 12 points and trim both ends
Z2 = np.linspace(0, 1, 12)[1:-1]
print(np.allclose(Z, Z2))                         # True

[0.09090909 0.18181818 0.27272727 0.36363636 0.45454545 0.54545455
 0.63636364 0.72727273 0.81818182 0.90909091]
10 True True
True


**Concept:** `np.linspace(start, stop, num, endpoint=False)` leaves out the stop value. Slicing off the first element removes 0, leaving 10 values strictly between 0 and 1. Alternatively, create 12 evenly spaced points and trim both ends with `[1:-1]`.

**Needed help?** solution  
**Revisit on:** Nov 8

---
### 40. Create a random vector of size 10 and sort it (★★☆)

In [11]:
rng = np.random.default_rng(42)
Z = rng.random(10)
print(Z)

print(np.sort(Z))               # returns a sorted COPY, Z unchanged
print(Z.argsort())              # indices that would sort Z

Z.sort()                        # sorts IN PLACE
print(Z)
print(Z[::-1])                  # descending order

[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735 0.97562235
 0.7611397  0.78606431 0.12811363 0.45038594]
[0.09417735 0.12811363 0.43887844 0.45038594 0.69736803 0.7611397
 0.77395605 0.78606431 0.85859792 0.97562235]
[4 8 1 9 3 6 0 7 2 5]
[0.09417735 0.12811363 0.43887844 0.45038594 0.69736803 0.7611397
 0.77395605 0.78606431 0.85859792 0.97562235]
[0.97562235 0.85859792 0.78606431 0.77395605 0.7611397  0.69736803
 0.45038594 0.43887844 0.12811363 0.09417735]


**Concept:** `np.sort(Z)` returns a sorted copy, while `Z.sort()` sorts the array in place and returns `None`. `Z.argsort()` returns the indices that would sort the array, which is useful for reordering other arrays. For descending order, reverse with `[::-1]`.

**Needed help?** no  
**Revisit on:** not needed